In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error = pd.read_csv('error_logs.csv')

# Parse timestamps as UTC datetimes (per rule 10)
for df in (metric, trace, log, error):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates
metric_cmdb = pd.Series(sorted(metric['cmdb_id'].dropna().unique()), name='cmdb_id')  # distinct cmdb_id values
metric_kpi_counts = (
    metric.groupby('kpi_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
metric_total_rows = metric.shape[0]

# 2) trace.csv aggregates
trace_kpi_counts = (
    trace.groupby('trace_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
trace_cmdb = pd.Series(sorted(trace['cmdb_id'].dropna().unique()), name='cmdb_id')
trace_total_rows = trace.shape[0]

# 3) log.csv aggregates
log_kpi_counts = (
    log.groupby('log_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
log_cmdb = pd.Series(sorted(log['cmdb_id'].dropna().unique()), name='cmdb_id')
log_total_rows = log.shape[0]

# 4) error_logs.csv aggregates
error_total_rows = error.shape[0]
error_cmdb = pd.Series(sorted(error['cmdb_id'].dropna().unique()), name='cmdb_id')
sample_messages = pd.Series(error['message'].dropna().unique()).head(5).reset_index(drop=True)

# Display compact summaries (multiple variables separated by commas as required)
metric_cmdb, metric_kpi_counts, metric_total_rows, trace_kpi_counts, trace_cmdb, trace_total_rows, log_kpi_counts, log_cmdb, log_total_rows, error_total_rows, error_cmdb, sample_messages

```
Out[1]:
```
summary = (
    "Summary:\n\n"
    "1) metric.csv\n"
    "- Distinct cmdb_id (15): carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, "
    "payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db.\n"
    "- Top kpi_name (kpi_name: count): socket:375, mem:375, cpu:372, diskio:200, workload:175, "
    "latency-50:147, latency-90:147, error:25.\n"
    "- Total metric rows: 1816.\n\n"
    "2) trace.csv\n"
    "- No trace data present. Total trace rows: 0. No trace_name or cmdb_id values found.\n\n"
    "3) log.csv\n"
    "- Top log_name (log_name: count): log.error_count:204, log.row_count:204.\n"
    "- Distinct cmdb_id (10): carts, carts-db, catalogue, front-end, orders, payment, queue-master, "
    "session-db, shipping, user.\n"
    "- Total log rows: 408.\n\n"
    "4) error_logs.csv\n"
    "- Total rows: 0. No cmdb_id values or sample messages present."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object,      kpi_name  count
0      socket    375
1         mem    375
2         cpu    372
3      diskio    200
4    workload    175
5  latency-50    147
6  latency-90    147
7       error     25, 1816, Empty DataFrame
Columns: [trace_name, count]
Index: [], Series([], Name: cmdb_id, dtype: object), 0,           log_name  count
0  log.error_count    204
1    log.row_count    204, 0           carts
1        carts-db
2       catalogue
3       front-end
4          orders
5         payment
6    queue-master
7      session-db
8        shipping
9            user
Name: cmdb_id, dtype: object, 408, 0, Series([], Name: cmdb_id, dtype: object), Series([], dtype: object))```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing 'metric' if present, else load
if 'metric' not in globals():
    metric = pd.read_csv('metric.csv')
# Ensure timestamp is parsed as UTC datetime
if metric['timestamp'].dtype != 'datetime64[ns, UTC]':
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire series (before filtering)
p95 = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# 2) Define incident window (UTC) and filter metric rows to that window
start = pd.to_datetime('2024-01-19 20:42:53', utc=True)
end = pd.to_datetime('2024-01-19 21:12:53', utc=True)
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# total number of metric rows inside the incident window
total_window_rows = metric_window.shape[0]

# If no rows in window, produce empty result table with specified columns
columns = [
    'cmdb_id', 'kpi_name', 'p95_threshold', 'num_points_in_window',
    'num_points_exceeding_p95', 'earliest_exceed_timestamp', 'max_value_in_window',
    'severity_ratio'
]

if total_window_rows == 0:
    result_df = pd.DataFrame(columns=columns)
else:
    # 3) Prepare window aggregates: count and max per (cmdb_id, kpi_name)
    window_agg = (
        metric_window
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(num_points_in_window=('value', 'size'),
             max_value_in_window=('value', 'max'))
        .reset_index()
    )

    # Merge p95 into window rows to evaluate exceedances per row
    mw = metric_window.merge(p95, on=['cmdb_id', 'kpi_name'], how='left')

    # Rows exceeding their global p95
    mw_exceed = mw[mw['value'] > mw['p95_threshold']]

    # Aggregate exceedance info per group
    if mw_exceed.shape[0] == 0:
        # No exceedances in the window
        result_df = pd.DataFrame(columns=columns)
    else:
        exceed_agg = (
            mw_exceed
            .groupby(['cmdb_id', 'kpi_name'])
            .agg(num_points_exceeding_p95=('value', 'size'),
                 earliest_exceed_timestamp=('timestamp', 'min'))
            .reset_index()
        )

        # Combine aggregates, p95, and compute severity ratio
        result_df = (
            exceed_agg
            .merge(window_agg, on=['cmdb_id', 'kpi_name'], how='left')
            .merge(p95, on=['cmdb_id', 'kpi_name'], how='left')
        )

        # Compute severity_ratio with safe divide (inf if p95==0 and max>0)
        result_df['severity_ratio'] = np.where(
            result_df['p95_threshold'] == 0,
            np.where(result_df['max_value_in_window'] > 0, np.inf, 0.0),
            result_df['max_value_in_window'] / result_df['p95_threshold']
        )

        # Reorder/select columns and sort by severity_ratio desc, limit to top 20
        result_df = result_df[
            ['cmdb_id', 'kpi_name', 'p95_threshold', 'num_points_in_window',
             'num_points_exceeding_p95', 'earliest_exceed_timestamp',
             'max_value_in_window', 'severity_ratio']
        ].sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# Display compact results: result_df and total_window_rows
result_df, total_window_rows

```
Out[2]:
```
summary = (
    "Incident window: 2024-01-19 20:42:53 → 2024-01-19 21:12:53 UTC\n"
    "Total metric rows in window: 1816\n\n"
    "Top (cmdb_id, kpi_name) pairs that exceeded their global P95 (sorted by severity_ratio, top 20):\n"
    "1) carts — cpu: p95=3.254, points=25, exceed=2, first_exceed=2024-01-19T20:53:00Z, max=7.389, severity=2.271\n"
    "2) catalogue — diskio: p95=42343.95, points=25, exceed=2, first_exceed=2024-01-19T20:45:00Z, max=75870.35, severity=1.792\n"
    "3) queue-master — diskio: p95=58474.31, points=25, exceed=2, first_exceed=2024-01-19T20:52:00Z, max=98136.66, severity=1.678\n"
    "4) shipping — latency-50: p95=0.003331, points=21, exceed=1, first_exceed=2024-01-19T20:49:00Z, max=0.004313, severity=1.295\n"
    "5) shipping — cpu: p95=1.299685, points=25, exceed=2, first_exceed=2024-01-19T20:55:00Z, max=1.662985, severity=1.280\n"
    "6) orders — cpu: p95=5.672168, points=25, exceed=2, first_exceed=2024-01-19T20:51:00Z, max=6.701475, severity=1.181\n"
    "7) shipping — latency-90: p95=0.006391, points=21, exceed=1, first_exceed=2024-01-19T20:49:00Z, max=0.007349, severity=1.150\n"
    "8) catalogue — latency-90: p95=0.008870, points=21, exceed=1, first_exceed=2024-01-19T21:08:00Z, max=0.010061, severity=1.134\n"
    "9) front-end — socket: p95=8.31, points=25, exceed=2, first_exceed=2024-01-19T20:58:00Z, max=9.216667, severity=1.109\n"
    "10) front-end — cpu: p95=4.861257, points=22, exceed=2, first_exceed=2024-01-19T20:50:00Z, max=5.309696, severity=1.092\n"
    "11) rabbitmq-exporter — cpu: p95=0.004804, points=25, exceed=2, first_exceed=2024-01-19T20:47:00Z, max=0.005235, severity=1.090\n"
    "12) queue-master — cpu: p95=0.943182, points=25, exceed=2, first_exceed=2024-01-19T20:54:00Z, max=1.007532, severity=1.068\n"
    "13) carts-db — cpu: p95=2.732994, points=25, exceed=2, first_exceed=2024-01-19T21:07:00Z, max=2.896281, severity=1.060\n"
    "14) front-end — latency-50: p95=0.030582, points=21, exceed=1, first_exceed=2024-01-19T20:49:00Z, max=0.032356, severity=1.058\n"
    "15) carts-db — diskio: p95=1,991,270.00, points=25, exceed=2, first_exceed=2024-01-19T21:02:00Z, max=2,102,070.00, severity=1.056\n"
    "16) shipping — socket: p95=6.613333, points=25, exceed=2, first_exceed=2024-01-19T20:54:00Z, max=6.933333, severity=1.048\n"
    "17) session-db — cpu: p95=0.299463, points=25, exceed=2, first_exceed=2024-01-19T21:08:00Z, max=0.313305, severity=1.046\n"
    "18) orders — workload: p95=2.185350, points=25, exceed=2, first_exceed=2024-01-19T21:02:00Z, max=2.256550, severity=1.033\n"
    "19) rabbitmq — cpu: p95=0.868675, points=25, exceed=2, first_exceed=2024-01-19T21:02:00Z, max=0.896187, severity=1.032\n"
    "20) carts — latency-50: p95=0.013825, points=21, exceed=1, first_exceed=2024-01-19T21:08:00Z, max=0.014165, severity=1.025\n\n"
    "Interpretation: 20 (cmdb_id, kpi) pairs exceeded their global P95 during the incident window. "
    "The largest relative breach was 'carts:cpu' (severity ~2.27× its global P95)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  p95_threshold  num_points_in_window  num_points_exceeding_p95 earliest_exceed_timestamp  max_value_in_window  severity_ratio
0               carts         cpu   3.253828e+00                    25                         2 2024-01-19 20:53:00+00:00         7.389310e+00        2.270959
1           catalogue      diskio   4.234395e+04                    25                         2 2024-01-19 20:45:00+00:00         7.587035e+04        1.791763
2        queue-master      diskio   5.847431e+04                    25                         2 2024-01-19 20:52:00+00:00         9.813666e+04        1.678287
3            shipping  latency-50   3.330970e-03                    21                         1 2024-01-19 20:49:00+00:00         4.313076e-03        1.294841
4            shipping         cpu   1.299685e+00                    25                         2 2024-01-19 20:55:00+00:00         1.662985e+00        1.279529
5              orders         cpu   5.672168e+00                    25                         2 2024-01-19 20:51:00+00:00         6.701475e+00        1.181466
6            shipping  latency-90   6.391028e-03                    21                         1 2024-01-19 20:49:00+00:00         7.348601e-03        1.149831
7           catalogue  latency-90   8.869627e-03                    21                         1 2024-01-19 21:08:00+00:00         1.006150e-02        1.134377
8           front-end      socket   8.310000e+00                    25                         2 2024-01-19 20:58:00+00:00         9.216667e+00        1.109105
9           front-end         cpu   4.861257e+00                    22                         2 2024-01-19 20:50:00+00:00         5.309696e+00        1.092248
10  rabbitmq-exporter         cpu   4.803599e-03                    25                         2 2024-01-19 20:47:00+00:00         5.235176e-03        1.089844
11       queue-master         cpu   9.431818e-01                    25                         2 2024-01-19 20:54:00+00:00         1.007532e+00        1.068226
12           carts-db         cpu   2.732994e+00                    25                         2 2024-01-19 21:07:00+00:00         2.896281e+00        1.059746
13          front-end  latency-50   3.058202e-02                    21                         1 2024-01-19 20:49:00+00:00         3.235560e-02        1.057994
14           carts-db      diskio   1.991270e+06                    25                         2 2024-01-19 21:02:00+00:00         2.102070e+06        1.055643
15           shipping      socket   6.613333e+00                    25                         2 2024-01-19 20:54:00+00:00         6.933333e+00        1.048387
16         session-db         cpu   2.994627e-01                    25                         2 2024-01-19 21:08:00+00:00         3.133048e-01        1.046223
17             orders    workload   2.185350e+00                    25                         2 2024-01-19 21:02:00+00:00         2.256550e+00        1.032581
18           rabbitmq         cpu   8.686747e-01                    25                         2 2024-01-19 21:02:00+00:00         8.961872e-01        1.031672
19              carts  latency-50   1.382533e-02                    21                         1 2024-01-19 21:08:00+00:00         1.416504e-02        1.024571, 1816)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing 'log' if present, else load
if 'log' not in globals():
    log = pd.read_csv('log.csv')

# Ensure timestamp is parsed as UTC datetime
if log['timestamp'].dtype != 'datetime64[ns, UTC]':
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, log_name) using the entire series (before filtering)
log_p95 = (
    log
    .groupby(['cmdb_id', 'log_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# 2) Define incident window (UTC) and filter log rows to that window
start = pd.to_datetime('2024-01-19 20:42:53', utc=True)
end = pd.to_datetime('2024-01-19 21:12:53', utc=True)
log_window = log[(log['timestamp'] >= start) & (log['timestamp'] <= end)].copy()

# total number of log rows inside the incident window
total_window_log_rows = log_window.shape[0]

# Prepare output columns
columns = [
    'cmdb_id', 'log_name', 'p95_threshold', 'num_points_in_window',
    'num_points_exceeding_p95', 'earliest_exceed_timestamp',
    'max_value_in_window', 'severity_ratio'
]

if total_window_log_rows == 0:
    result_log_df = pd.DataFrame(columns=columns)
else:
    # Window aggregates: count and max per (cmdb_id, log_name)
    log_window_agg = (
        log_window
        .groupby(['cmdb_id', 'log_name'])
        .agg(num_points_in_window=('value', 'size'),
             max_value_in_window=('value', 'max'))
        .reset_index()
    )

    # Merge global p95 into window rows to evaluate exceedances per row
    lw = log_window.merge(log_p95, on=['cmdb_id', 'log_name'], how='left')

    # Rows exceeding their global p95
    lw_exceed = lw[lw['value'] > lw['p95_threshold']]

    if lw_exceed.shape[0] == 0:
        result_log_df = pd.DataFrame(columns=columns)
    else:
        exceed_agg = (
            lw_exceed
            .groupby(['cmdb_id', 'log_name'])
            .agg(num_points_exceeding_p95=('value', 'size'),
                 earliest_exceed_timestamp=('timestamp', 'min'))
            .reset_index()
        )

        # Combine aggregates, p95, compute severity ratio
        result_log_df = (
            exceed_agg
            .merge(log_window_agg, on=['cmdb_id', 'log_name'], how='left')
            .merge(log_p95, on=['cmdb_id', 'log_name'], how='left')
        )

        result_log_df['severity_ratio'] = np.where(
            result_log_df['p95_threshold'] == 0,
            np.where(result_log_df['max_value_in_window'] > 0, np.inf, 0.0),
            result_log_df['max_value_in_window'] / result_log_df['p95_threshold']
        )

        result_log_df = result_log_df[
            ['cmdb_id', 'log_name', 'p95_threshold', 'num_points_in_window',
             'num_points_exceeding_p95', 'earliest_exceed_timestamp',
             'max_value_in_window', 'severity_ratio']
        ].sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# Display compact results: result_log_df and total_window_log_rows
result_log_df, total_window_log_rows

```
Out[3]:
```
Incident window: 2024-01-19 20:42:53 → 2024-01-19 21:12:53 UTC
Total log rows in window: 408

(log entries in the window that exceeded their global P95 — sorted by severity_ratio)

1) user — log.row_count: p95=613.8, points=25, exceed=2, first_exceed=2024-01-19T20:52:00Z, max=637, severity=1.038  
2) catalogue — log.row_count: p95=172.4, points=25, exceed=2, first_exceed=2024-01-19T20:49:00Z, max=177, severity=1.027  
3) front-end — log.row_count: p95=2051.5, points=22, exceed=2, first_exceed=2024-01-19T21:00:00Z, max=2101, severity=1.024  
4) orders — log.row_count: p95=132.0, points=21, exceed=1, first_exceed=2024-01-19T21:08:00Z, max=134, severity=1.015  
5) queue-master — log.row_count: p95=594.0, points=21, exceed=1, first_exceed=2024-01-19T21:08:00Z, max=603, severity=1.015  
6) shipping — log.row_count: p95=66.0, points=21, exceed=1, first_exceed=2024-01-19T21:08:00Z, max=67, severity=1.015  
7) payment — log.row_count: p95=105.8, points=25, exceed=2, first_exceed=2024-01-19T21:05:00Z, max=107, severity=1.011

Interpretation: Seven (cmdb_id, log_name) pairs exceeded their global P95 during the incident window. The largest relative breach was user:log.row_count (~1.038× its global P95).

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id       log_name  p95_threshold  num_points_in_window  num_points_exceeding_p95 earliest_exceed_timestamp  max_value_in_window  severity_ratio
0          user  log.row_count          613.8                    25                         2 2024-01-19 20:52:00+00:00                  637        1.037797
1     catalogue  log.row_count          172.4                    25                         2 2024-01-19 20:49:00+00:00                  177        1.026682
2     front-end  log.row_count         2051.5                    22                         2 2024-01-19 21:00:00+00:00                 2101        1.024129
3        orders  log.row_count          132.0                    21                         1 2024-01-19 21:08:00+00:00                  134        1.015152
4  queue-master  log.row_count          594.0                    21                         1 2024-01-19 21:08:00+00:00                  603        1.015152
5      shipping  log.row_count           66.0                    21                         1 2024-01-19 21:08:00+00:00                   67        1.015152
6       payment  log.row_count          105.8                    25                         2 2024-01-19 21:05:00+00:00                  107        1.011342, 408)```
```